# Appendix — Radiomics pipeline demo từ ảnh đến CSV

Trang này là phần nối giữa dữ liệu ảnh và bảng feature. Mục tiêu không phải thay thế pipeline thật của nghiên cứu, mà giúp học sinh hiểu vì sao cuối cùng lại có CSV radiomics để đưa vào Day 01–Day 05.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
plt.rcParams["figure.figsize"] = (5,5)

## 1. Tạo CT giả lập và mask khối u

Dùng mảng số để mô phỏng một lát cắt CT.

In [ ]:

np.random.seed(42)
img = np.random.normal(-500, 80, size=(128, 128))
y, x = np.ogrid[:128, :128]
center = (64, 64)
r = 18
mask = ((x-center[1])**2 + (y-center[0])**2) <= r**2
img[mask] += 120

fig, axes = plt.subplots(1, 2, figsize=(8,4))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("CT giả lập")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Mask khối u")
plt.tight_layout()
plt.show()


## 2. HU clipping

In [ ]:

img_clip = np.clip(img, -800, 800)
img_clip.min(), img_clip.max()


## 3. Tạo ring 1, 3, 5 mm bằng distance transform

Ví dụ dưới đây chỉ mô phỏng trên ảnh 2D với giả định voxel 1 mm.

In [ ]:

dil1 = ndi.binary_dilation(mask, iterations=1)
dil3 = ndi.binary_dilation(mask, iterations=3)
dil5 = ndi.binary_dilation(mask, iterations=5)
ring1 = dil1 ^ mask
ring3 = dil3 ^ mask
ring5 = dil5 ^ mask

fig, axes = plt.subplots(1, 4, figsize=(12,3))
for ax, arr, title in zip(axes, [mask, ring1, ring3, ring5], ["intra", "ring1", "ring3", "ring5"]):
    ax.imshow(arr, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Tính vài feature đơn giản để minh họa ý tưởng

In [ ]:

def simple_features(image, roi):
    vals = image[roi]
    return {
        "meanHU": float(vals.mean()),
        "medianHU": float(np.median(vals)),
        "stdHU": float(vals.std(ddof=1)),
        "n_voxels": int(vals.size)
    }

rows = []
for name, roi in [("intra", mask), ("ring1", ring1), ("ring3", ring3), ("ring5", ring5)]:
    feat = simple_features(img_clip, roi)
    feat["roi"] = name
    rows.append(feat)

pd.DataFrame(rows)


## 5. Ý nghĩa của appendix này

Luồng tổng quát là:

1. ảnh CT và mask  
2. tiền xử lý như resample, HU clip  
3. tạo intra và ring  
4. trích xuất đặc trưng  
5. ghép lại thành CSV radiomics  
6. dùng CSV đó cho các notebook phân tích phía sau

Đây chính là cầu nối từ ảnh `.nii.gz` sang các bảng `features_HUclip_intra.csv`, `features_HUclip_ring1.csv`, `features_HUclip_ring3.csv`, `features_HUclip_ring5.csv` trong phân tích thật.